# 01. Setup & Baseline (Text Cell)

ADR-016 (분류) + ADR-017 (회귀) Phase 2 진입.

- 6 dataset 로드 + clean DSC baseline 측정 (default 가중치, ADR-015 fallback)
- 결과: `results/text_baseline_dsc.csv`

Phase 2 후속 노트북:
- 02 polluter sweep, 03 model train, 04 분석


In [ ]:
# ============================================================
# 0. Drive 마운트 + dsc/ 자동 검색 + sys.path 등록
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, glob, json
import numpy as np
import pandas as pd


def _find_dsc_base():
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py',
                f'{root}/*/*/dsc_framework/__init__.py',
                f'{root}/*/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    drive_root = '/content/drive/MyDrive'
    listing = os.listdir(drive_root) if os.path.isdir(drive_root) else []
    raise RuntimeError(
        'dsc_framework/ 폴더를 G드라이브에서 못 찾음.\n'
        '  1) G드라이브 클라이언트 sync 완료 확인 (commit 직후면 잠시 대기 후 재시도)\n'
        '  2) Drive 마운트 확인 — !ls /content/drive/MyDrive\n'
        f'  현재 Drive 내용: {listing[:20]}'
    )

# 누락 파일 진단 — partial sync 시 빠른 실패
REQUIRED = ['shared_metrics.py', 'classification_cell.py', 'regression_cell.py',
            'image_cell.py', 'text_cell.py', 'text_cell_regression.py',
            'text_trainers.py', 'data_type_detection.py', 'router.py',
            'text_polluters', 'image_polluters']
missing = [f for f in REQUIRED if not os.path.exists(f'{BASE}/dsc_framework/{f}')]
if missing:
    raise RuntimeError(
        f'dsc_framework/ 파일 누락: {missing}\n'
        '→ G드라이브 sync 미완료. 잠시 대기 후 재실행.\n'
        '→ Colab Drive view stale 시: drive.flush_and_unmount() 후 재마운트.'
    )

RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/text'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

print(f'BASE: {BASE}')
print(f'dsc_framework 파일: {sorted(f for f in os.listdir(f"{BASE}/dsc_framework") if not f.startswith("_"))}')


In [ ]:
# ============================================================
# 의존성 설치 (Colab 1회 실행 후 다음 셀)
# ============================================================
%pip install -q 'transformers>=4.30' 'datasets>=2.10' 'xgboost>=1.7' 'accelerate>=1.1.0'


In [ ]:
# ============================================================
# Restart 권장 — pip install 후 import 안정성 위해 런타임 한 번 재시작
# 또는 force reimport.
# ============================================================
from dsc_framework.text_cell import compute_dsc_text
from dsc_framework.text_cell_regression import compute_dsc_text_regression
print('text cell modules OK')


In [ ]:
# ============================================================
# 데이터셋 로드 (ADR-016 분류 3종 + ADR-017 회귀 3종)
#   - Phase 2 정식 실행 시 N_TRAIN/N_TEST를 ADR §4 sample_cap으로 키울 것
#   - sanity/dev에선 작은 sample로 시작
# ============================================================
from datasets import load_dataset
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

# 사용자가 sample size 조정. ADR §4 정식 = train 50000~200000 / test 5000~50000
N_TRAIN = 3000  # 분류·SST5는 자동 cap, 빈 dataset이면 자체 train size
N_TEST  = 500


def _slice(ds_dict, split, n, seed=42):
    ds = ds_dict[split] if split in ds_dict else ds_dict['train']
    if n is None or len(ds) <= n:
        return ds
    return ds.shuffle(seed=seed).select(range(n))


def load_all():
    """분류 3 + 회귀 3 = 6 dataset을 (tr_texts, tr_y, te_texts, te_y, task)로 반환."""
    out = {}

    # 분류
    ag = load_dataset('fancyzhx/ag_news')
    out['ag_news'] = (_slice(ag, 'train', N_TRAIN), _slice(ag, 'test', N_TEST), 'classification')

    imdb = load_dataset('stanfordnlp/imdb')
    out['imdb'] = (_slice(imdb, 'train', N_TRAIN), _slice(imdb, 'test', N_TEST), 'classification')

    news20 = load_dataset('SetFit/20_newsgroups')
    out['20news'] = (_slice(news20, 'train', N_TRAIN), _slice(news20, 'test', N_TEST), 'classification')

    # 회귀 (label = star/sentiment를 float)
    yelp = load_dataset('Yelp/yelp_review_full')
    out['yelp_full'] = (_slice(yelp, 'train', N_TRAIN), _slice(yelp, 'test', N_TEST), 'regression')

    amazon = load_dataset('SetFit/amazon_reviews_multi_en')  # ADR-017 미러
    out['amazon_en'] = (_slice(amazon, 'train', N_TRAIN), _slice(amazon, 'test', N_TEST), 'regression')

    sst = load_dataset('SetFit/sst5')
    out['sst5'] = (_slice(sst, 'train', N_TRAIN), _slice(sst, 'test', N_TEST), 'regression')

    return out


def to_lists(ds_split, task):
    texts = ds_split['text']
    labels = ds_split['label']
    if task == 'regression':
        labels = [float(y) for y in labels]
    return list(texts), list(labels)


print('load_dataset OK — load_all() 호출하면 6 dataset 로드 시작.')


In [ ]:
# ============================================================
# 6 dataset 로드 + clean DSC baseline 측정
# ============================================================
datasets = load_all()

rows = []
for name, (tr_ds, te_ds, task) in datasets.items():
    texts, labels = to_lists(tr_ds, task)
    print(f'\n=== {name} ({task}, n={len(texts)}) ===')
    if task == 'classification':
        r = compute_dsc_text(texts, labels, use_embeddings=True,
                             sample_cap=1000, random_state=42)
    else:
        r = compute_dsc_text_regression(texts, labels, use_embeddings=True,
                                        sample_cap=1000, random_state=42)
    print(f'  DSC={r["score"]} ({r["grade"]})')
    row = {'dataset': name, 'task': task, 'n': len(texts), **{k: v for k, v in r.items()}}
    rows.append(row)

baseline = pd.DataFrame(rows)
out_path = f'{RESULTS_DIR}/text_baseline_dsc.csv'
baseline.to_csv(out_path, index=False)
print(f'\nbaseline saved: {out_path}')
baseline


---

다음: `02_pollution_and_dsc_text.ipynb` — polluter 스윕.
